In [ ]:
# from pathlib import Path
# import gc
# import json
# import os
# import time
# from typing import List

# import numpy as np
# import pandas as pd
# import soundfile as sf
# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# from tqdm.auto import tqdm

# from transformers import AutoProcessor, Qwen2AudioForConditionalGeneration
# from comet import download_model, load_from_checkpoint
# from numcodecs import Quantize, Zstd


# # ============================================================
# # CONFIG
# # ============================================================

# PROJECT_ROOT = Path("/home/paperspace/projects/iwslt2026-compression")
# OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "experiment_results"

# MODEL_ID = "Qwen/Qwen2-Audio-7B-Instruct"
# PROMPT_TEXT = "Translate this English speech into Chinese."
# MAX_NEW_TOKENS = 64
# COMET_BATCH_SIZE = 16
# EVAL_MANIFEST = PROJECT_ROOT / "data" / "manifests" / "acl6060_eval_en_zh.csv"

# # Main knobs for the next variation sweep
# CODEC_DIGITS = int(os.environ.get("CODEC_DIGITS", "3"))
# CODEC_ZSTD_LEVEL = int(os.environ.get("CODEC_ZSTD_LEVEL", "3"))
# LAYER_SUBSET = os.environ.get("LAYER_SUBSET", "all_selected")
# # allowed: all_selected, up_down_only, down_only, up_only, gate_only

# EXPERIMENT_NAME = f"custom_selected_linear_codec_{LAYER_SUBSET}_q{CODEC_DIGITS}_zstd_eval_only_en_zh"
# EXP_DIR = OUTPUT_ROOT / EXPERIMENT_NAME
# EXP_DIR.mkdir(parents=True, exist_ok=True)

# print("project root exists:", PROJECT_ROOT.exists())
# print("cuda available:", torch.cuda.is_available())
# if torch.cuda.is_available():
#     print("gpu:", torch.cuda.get_device_name(0))
# print("experiment:", EXPERIMENT_NAME)
# print("codec digits:", CODEC_DIGITS)
# print("layer subset:", LAYER_SUBSET)


# # ============================================================
# # DATA / PROCESSOR / COMET
# # ============================================================

# eval_df = pd.read_csv(EVAL_MANIFEST).copy()
# eval_df["audio_path"] = eval_df["audio_path"].apply(lambda p: str((PROJECT_ROOT / p).resolve()))

# processor = AutoProcessor.from_pretrained(MODEL_ID)

# comet_path = download_model("Unbabel/wmt22-comet-da")
# comet_model = load_from_checkpoint(comet_path)

# print("eval rows:", len(eval_df))


# # ============================================================
# # CODEC HELPERS
# # ============================================================

# class CodecPipeline:
#     def __init__(self, filters, compressor=None, encoded_dtype="f2"):
#         self.filters = filters
#         self.compressor = compressor
#         self.encoded_dtype = np.dtype(encoded_dtype)

#     def encode(self, arr):
#         x = np.asarray(arr, dtype=np.float32)
#         for f in self.filters:
#             x = f.encode(x)
#         x = np.asarray(x, dtype=self.encoded_dtype)
#         filtered_shape = x.shape
#         filtered_bytes = x.tobytes(order="C")
#         if self.compressor is not None:
#             filtered_bytes = self.compressor.encode(filtered_bytes)
#         return {
#             "payload": filtered_bytes,
#             "filtered_shape": filtered_shape,
#             "filtered_dtype": self.encoded_dtype.str,
#         }

#     def decode(self, encoded_obj, original_shape, original_dtype=np.float32):
#         payload = encoded_obj["payload"]
#         filtered_shape = tuple(encoded_obj["filtered_shape"])
#         filtered_dtype = np.dtype(encoded_obj["filtered_dtype"])
#         if self.compressor is not None:
#             payload = self.compressor.decode(payload)
#         x = np.frombuffer(payload, dtype=filtered_dtype).copy().reshape(filtered_shape)
#         for f in reversed(self.filters):
#             x = f.decode(x)
#         return np.asarray(x, dtype=original_dtype).reshape(original_shape)


# class CompressedLinear(nn.Module):
#     def __init__(self, linear_layer, pipeline, description):
#         super().__init__()
#         weight = linear_layer.weight.detach().cpu().numpy().astype(np.float32)
#         self.weight_shape = weight.shape
#         self.in_features = linear_layer.in_features
#         self.out_features = linear_layer.out_features
#         self.pipeline = pipeline
#         self.description = description
#         self.encoded_weight = self.pipeline.encode(weight)

#         if linear_layer.bias is not None:
#             self.bias = nn.Parameter(linear_layer.bias.detach().clone(), requires_grad=False)
#         else:
#             self.bias = None

#         self._cached_weight = None
#         self._cached_device = None
#         self._cached_dtype = None

#     def _get_cached_weight(self, device, dtype):
#         if self._cached_weight is None or self._cached_device != device or self._cached_dtype != dtype:
#             decoded = self.pipeline.decode(self.encoded_weight, self.weight_shape, np.float32)
#             self._cached_weight = torch.tensor(decoded, device=device, dtype=dtype)
#             self._cached_device = device
#             self._cached_dtype = dtype
#         return self._cached_weight

#     def forward(self, x):
#         weight = self._get_cached_weight(x.device, x.dtype)
#         return F.linear(x, weight, self.bias)


# # ============================================================
# # MODEL HELPERS
# # ============================================================

# def get_module_by_name(model, module_name):
#     module = model
#     for part in module_name.split("."):
#         module = getattr(module, part)
#     return module


# def set_module_by_name(model, module_name, new_module):
#     parts = module_name.split(".")
#     parent = model
#     for part in parts[:-1]:
#         parent = getattr(parent, part)
#     setattr(parent, parts[-1], new_module)


# def get_linear_layer_names(model) -> List[str]:
#     return [name for name, module in model.named_modules() if isinstance(module, nn.Linear)]


# def should_select_layer(layer_name: str) -> bool:
#     return any(pattern in layer_name for pattern in ["mlp", "feed_forward", "up_proj", "down_proj", "gate_proj"])


# def should_keep_for_subset(layer_name: str, subset: str) -> bool:
#     if subset == "all_selected":
#         return should_select_layer(layer_name)
#     if subset == "up_down_only":
#         return ("up_proj" in layer_name) or ("down_proj" in layer_name)
#     if subset == "down_only":
#         return "down_proj" in layer_name
#     if subset == "up_only":
#         return "up_proj" in layer_name
#     if subset == "gate_only":
#         return "gate_proj" in layer_name
#     raise ValueError(f"Unsupported LAYER_SUBSET={subset}")


# def get_selected_linear_layer_names(model, subset: str = LAYER_SUBSET) -> List[str]:
#     return [name for name in get_linear_layer_names(model) if should_keep_for_subset(name, subset)]


# def estimate_linear_storage_bytes(layer) -> int:
#     if isinstance(layer, nn.Linear):
#         total = layer.weight.detach().cpu().numpy().nbytes
#         if layer.bias is not None:
#             total += layer.bias.detach().cpu().numpy().nbytes
#         return total
#     if isinstance(layer, CompressedLinear):
#         total = len(layer.encoded_weight["payload"])
#         if layer.bias is not None:
#             total += layer.bias.detach().cpu().numpy().nbytes
#         return total
#     return 0


# def cleanup_cuda():
#     gc.collect()
#     if torch.cuda.is_available():
#         torch.cuda.empty_cache()


# # ============================================================
# # INFERENCE HELPERS
# # ============================================================

# def generate_translation(model, processor, audio_path, prompt_text=PROMPT_TEXT, max_new_tokens=MAX_NEW_TOKENS):
#     audio, sr = sf.read(str(audio_path))
#     if audio.ndim > 1:
#         audio = audio.mean(axis=1)

#     conversation = [
#         {
#             "role": "user",
#             "content": [
#                 {"type": "audio", "audio_url": str(audio_path)},
#                 {"type": "text", "text": prompt_text},
#             ],
#         }
#     ]

#     text = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
#     inputs = processor(text=text, audios=[audio], sampling_rate=sr, return_tensors="pt")
#     model_device = next(model.parameters()).device
#     inputs = {k: v.to(model_device) if hasattr(v, "to") else v for k, v in inputs.items()}

#     with torch.no_grad():
#         generated_ids = model.generate(
#             **inputs,
#             max_new_tokens=max_new_tokens,
#             do_sample=False,
#             num_beams=1,
#             use_cache=True,
#         )

#     generated_ids = generated_ids[:, inputs["input_ids"].size(1):]
#     pred = processor.batch_decode(generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0].strip()
#     return pred


# def run_inference(model, df_input, run_name, max_new_tokens=MAX_NEW_TOKENS):
#     preds = []
#     start = time.time()
#     for _, row in tqdm(df_input.iterrows(), total=len(df_input), desc=run_name):
#         pred = generate_translation(model, processor, Path(row["audio_path"]), max_new_tokens=max_new_tokens)
#         preds.append(pred)
#     elapsed = time.time() - start
#     out_df = df_input.copy()
#     out_df["prediction"] = preds
#     return out_df, elapsed


# def compute_comet(pred_df, comet_model, batch_size=COMET_BATCH_SIZE):
#     comet_data = [
#         {"src": r["source_en"], "mt": r["prediction"], "ref": r["target_zh"]}
#         for _, r in pred_df.iterrows()
#     ]
#     comet_out = comet_model.predict(comet_data, batch_size=batch_size, gpus=1)
#     return comet_out.system_score if hasattr(comet_out, "system_score") else comet_out[1]


# # ============================================================
# # APPLY FIXED SELECTED-LAYER CODEC
# # ============================================================

# def make_pipeline_for_run():
#     return CodecPipeline(
#         filters=[Quantize(digits=CODEC_DIGITS, dtype="f4", astype="f2")],
#         compressor=Zstd(level=CODEC_ZSTD_LEVEL),
#         encoded_dtype="f2",
#     )


# def apply_selected_layer_codec(model, subset: str = LAYER_SUBSET):
#     selected_layers = get_selected_linear_layer_names(model, subset=subset)
#     pipeline = make_pipeline_for_run()
#     desc = f"Selected-layer codec: subset={subset}, Quantize(digits={CODEC_DIGITS}, astype='f2') + Zstd"

#     applied_rows = []
#     for layer_name in tqdm(selected_layers, desc="apply_selected_layer_codec"):
#         layer = get_module_by_name(model, layer_name)
#         if not isinstance(layer, nn.Linear):
#             continue
#         compressed_layer = CompressedLinear(layer, pipeline, desc)
#         compressed_layer = compressed_layer.to(next(model.parameters()).device)
#         set_module_by_name(model, layer_name, compressed_layer)
#         applied_rows.append({
#             "layer_name": layer_name,
#             "layer_subset": subset,
#             "compression_method": desc,
#         })
#     return selected_layers, pd.DataFrame(applied_rows)


# # ============================================================
# # LOAD MODEL
# # ============================================================

# baseline_model = Qwen2AudioForConditionalGeneration.from_pretrained(
#     MODEL_ID,
#     torch_dtype=torch.float16,
#     device_map="auto",
# )
# baseline_model.eval()


# # ============================================================
# # RECORD ORIGINAL STORAGE
# # ============================================================

# original_linear_bytes = {}
# for layer_name in get_linear_layer_names(baseline_model):
#     layer = get_module_by_name(baseline_model, layer_name)
#     if isinstance(layer, nn.Linear):
#         original_linear_bytes[layer_name] = estimate_linear_storage_bytes(layer)


# # ============================================================
# # APPLY COMPRESSION
# # ============================================================

# selected_layers, applied_df = apply_selected_layer_codec(baseline_model, subset=LAYER_SUBSET)

# applied_path = EXP_DIR / "selected_layer_codec_policy.csv"
# applied_df.to_csv(applied_path, index=False)
# selected_layers_path = EXP_DIR / "selected_layers.json"
# with open(selected_layers_path, "w", encoding="utf-8") as f:
#     json.dump(selected_layers, f, indent=2)

# print("selected layers count:", len(selected_layers))
# print("policy saved to:", applied_path)


# # ============================================================
# # STORAGE REPORT
# # ============================================================

# storage_rows = []
# total_original = 0
# total_compressed = 0

# for layer_name in get_linear_layer_names(baseline_model):
#     layer = get_module_by_name(baseline_model, layer_name)
#     orig_bytes = original_linear_bytes.get(layer_name, np.nan)
#     comp_bytes = estimate_linear_storage_bytes(layer)

#     if not np.isnan(orig_bytes):
#         total_original += orig_bytes
#     total_compressed += comp_bytes

#     storage_rows.append(
#         {
#             "layer_name": layer_name,
#             "is_selected_layer": layer_name in set(selected_layers),
#             "original_bytes": orig_bytes,
#             "compressed_bytes": comp_bytes,
#             "compression_ratio": (orig_bytes / comp_bytes) if (not np.isnan(orig_bytes) and comp_bytes > 0) else np.nan,
#             "method": getattr(layer, "description", "original"),
#             "module_type": type(layer).__name__,
#         }
#     )

# storage_report_df = pd.DataFrame(storage_rows)
# storage_report_path = EXP_DIR / "selected_layer_codec_storage_report.csv"
# storage_report_df.to_csv(storage_report_path, index=False)

# storage_totals = {
#     "original_total_bytes": total_original,
#     "compressed_total_bytes": total_compressed,
#     "total_ratio": total_original / total_compressed if total_compressed > 0 else np.nan,
# }

# print("storage report saved to:", storage_report_path)
# print(storage_totals)


# # ============================================================
# # EVAL
# # ============================================================

# experiment_model = baseline_model
# experiment_model.eval()

# pred_df, eval_time = run_inference(
#     experiment_model,
#     eval_df,
#     run_name=f"selected_layer_codec_{LAYER_SUBSET}_q{CODEC_DIGITS}_eval",
#     max_new_tokens=MAX_NEW_TOKENS,
# )

# preds_path = EXP_DIR / "selected_layer_codec_eval_preds.csv"
# pred_df.to_csv(preds_path, index=False, encoding="utf-8")
# comet_score = compute_comet(pred_df, comet_model)

# print("predictions saved to:", preds_path)
# print("eval seconds:", round(eval_time, 2))
# print("eval COMET:", comet_score)


# # ============================================================
# # SUMMARY
# # ============================================================

# timestamp_utc = pd.Timestamp.utcnow().isoformat()

# summary_df = pd.DataFrame([
#     {
#         "timestamp_utc": timestamp_utc,
#         "experiment_name": EXPERIMENT_NAME,
#         "run": f"selected_layer_codec_{LAYER_SUBSET}_q{CODEC_DIGITS}_eval",
#         "split": "eval",
#         "rows": len(pred_df),
#         "seconds": eval_time,
#         "seconds_per_item": eval_time / len(pred_df),
#         "comet": comet_score,
#         "model_id": MODEL_ID,
#         "prompt_text": PROMPT_TEXT,
#         "max_new_tokens": MAX_NEW_TOKENS,
#         "comet_batch_size": COMET_BATCH_SIZE,
#         "predictions_csv": str(preds_path),
#         "quantization_method": "custom_selected_linear_codec",
#         "precision": "custom",
#         "layer_subset": LAYER_SUBSET,
#         "codec_digits": CODEC_DIGITS,
#         "codec_zstd_level": CODEC_ZSTD_LEVEL,
#         "selected_layer_count": len(selected_layers),
#         "original_total_bytes": storage_totals["original_total_bytes"],
#         "compressed_total_bytes": storage_totals["compressed_total_bytes"],
#         "compression_ratio": storage_totals["total_ratio"],
#         "compression_method": f"Selected-layer codec subset={LAYER_SUBSET} q{CODEC_DIGITS} + Zstd",
#         "selected_layers_json": str(selected_layers_path),
#         "storage_report_csv": str(storage_report_path),
#     }
# ])

# summary_path = EXP_DIR / "selected_layer_codec_eval_summary.csv"
# summary_df.to_csv(summary_path, index=False)

# print("summary saved to:", summary_path)
# print(summary_df)


# # ============================================================
# # CLEANUP
# # ============================================================

# del experiment_model
# cleanup_cuda()


In [1]:
from pathlib import Path
import gc
import json
import os
import time
from typing import List

import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

from transformers import AutoProcessor, Qwen2AudioForConditionalGeneration
from comet import download_model, load_from_checkpoint
from numcodecs import Quantize, Zstd


# ============================================================
# CONFIG
# ============================================================

PROJECT_ROOT = Path("/home/paperspace/projects/iwslt2026-compression")
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "experiment_results"

MODEL_ID = "Qwen/Qwen2-Audio-7B-Instruct"
PROMPT_TEXT = "Translate this English speech into Chinese."
MAX_NEW_TOKENS = 64
COMET_BATCH_SIZE = 16
EVAL_MANIFEST = PROJECT_ROOT / "data" / "manifests" / "acl6060_eval_en_zh.csv"

# Main knobs for the next variation sweep
CODEC_DIGITS = int(os.environ.get("CODEC_DIGITS", "3"))
CODEC_ZSTD_LEVEL = int(os.environ.get("CODEC_ZSTD_LEVEL", "3"))
LAYER_SUBSET = os.environ.get("LAYER_SUBSET", "all_selected")
# allowed: all_selected, up_down_only, down_only, up_only, gate_only

EXPERIMENT_NAME = f"custom_selected_linear_codec_{LAYER_SUBSET}_q{CODEC_DIGITS}_zstd_eval_only_en_zh"
EXP_DIR = OUTPUT_ROOT / EXPERIMENT_NAME
EXP_DIR.mkdir(parents=True, exist_ok=True)

print("project root exists:", PROJECT_ROOT.exists())
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
print("experiment:", EXPERIMENT_NAME)
print("codec digits:", CODEC_DIGITS)
print("layer subset:", LAYER_SUBSET)


# ============================================================
# DATA / PROCESSOR / COMET
# ============================================================

eval_df = pd.read_csv(EVAL_MANIFEST).copy()
eval_df["audio_path"] = eval_df["audio_path"].apply(lambda p: str((PROJECT_ROOT / p).resolve()))

processor = AutoProcessor.from_pretrained(MODEL_ID)

comet_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(comet_path)

print("eval rows:", len(eval_df))


# ============================================================
# CODEC HELPERS
# ============================================================

class CodecPipeline:
    def __init__(self, filters, compressor=None, encoded_dtype="f2"):
        self.filters = filters
        self.compressor = compressor
        self.encoded_dtype = np.dtype(encoded_dtype)

    def encode(self, arr):
        x = np.asarray(arr, dtype=np.float32)
        for f in self.filters:
            x = f.encode(x)
        x = np.asarray(x, dtype=self.encoded_dtype)
        filtered_shape = x.shape
        filtered_bytes = x.tobytes(order="C")
        if self.compressor is not None:
            filtered_bytes = self.compressor.encode(filtered_bytes)
        return {
            "payload": filtered_bytes,
            "filtered_shape": filtered_shape,
            "filtered_dtype": self.encoded_dtype.str,
        }

    def decode(self, encoded_obj, original_shape, original_dtype=np.float32):
        payload = encoded_obj["payload"]
        filtered_shape = tuple(encoded_obj["filtered_shape"])
        filtered_dtype = np.dtype(encoded_obj["filtered_dtype"])
        if self.compressor is not None:
            payload = self.compressor.decode(payload)
        x = np.frombuffer(payload, dtype=filtered_dtype).copy().reshape(filtered_shape)
        for f in reversed(self.filters):
            x = f.decode(x)
        return np.asarray(x, dtype=original_dtype).reshape(original_shape)


class CompressedLinear(nn.Module):
    def __init__(self, linear_layer, pipeline, description):
        super().__init__()
        weight = linear_layer.weight.detach().cpu().numpy().astype(np.float32)
        self.weight_shape = weight.shape
        self.in_features = linear_layer.in_features
        self.out_features = linear_layer.out_features
        self.pipeline = pipeline
        self.description = description
        self.encoded_weight = self.pipeline.encode(weight)

        if linear_layer.bias is not None:
            self.bias = nn.Parameter(linear_layer.bias.detach().clone(), requires_grad=False)
        else:
            self.bias = None

        self._cached_weight = None
        self._cached_device = None
        self._cached_dtype = None

    def _get_cached_weight(self, device, dtype):
        if self._cached_weight is None or self._cached_device != device or self._cached_dtype != dtype:
            decoded = self.pipeline.decode(self.encoded_weight, self.weight_shape, np.float32)
            self._cached_weight = torch.tensor(decoded, device=device, dtype=dtype)
            self._cached_device = device
            self._cached_dtype = dtype
        return self._cached_weight

    def forward(self, x):
        weight = self._get_cached_weight(x.device, x.dtype)
        return F.linear(x, weight, self.bias)


# ============================================================
# MODEL HELPERS
# ============================================================

def get_module_by_name(model, module_name):
    module = model
    for part in module_name.split("."):
        module = getattr(module, part)
    return module


def set_module_by_name(model, module_name, new_module):
    parts = module_name.split(".")
    parent = model
    for part in parts[:-1]:
        parent = getattr(parent, part)
    setattr(parent, parts[-1], new_module)


def get_linear_layer_names(model) -> List[str]:
    return [name for name, module in model.named_modules() if isinstance(module, nn.Linear)]


def should_select_layer(layer_name: str) -> bool:
    return any(pattern in layer_name for pattern in ["mlp", "feed_forward", "up_proj", "down_proj", "gate_proj"])


def should_keep_for_subset(layer_name: str, subset: str) -> bool:
    if subset == "all_selected":
        return should_select_layer(layer_name)
    if subset == "up_down_only":
        return ("up_proj" in layer_name) or ("down_proj" in layer_name)
    if subset == "down_only":
        return "down_proj" in layer_name
    if subset == "up_only":
        return "up_proj" in layer_name
    if subset == "gate_only":
        return "gate_proj" in layer_name
    raise ValueError(f"Unsupported LAYER_SUBSET={subset}")


def get_selected_linear_layer_names(model, subset: str = LAYER_SUBSET) -> List[str]:
    return [name for name in get_linear_layer_names(model) if should_keep_for_subset(name, subset)]


def estimate_linear_storage_bytes(layer) -> int:
    if isinstance(layer, nn.Linear):
        total = layer.weight.detach().cpu().numpy().nbytes
        if layer.bias is not None:
            total += layer.bias.detach().cpu().numpy().nbytes
        return total
    if isinstance(layer, CompressedLinear):
        total = len(layer.encoded_weight["payload"])
        if layer.bias is not None:
            total += layer.bias.detach().cpu().numpy().nbytes
        return total
    return 0


def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ============================================================
# INFERENCE HELPERS
# ============================================================

def generate_translation(model, processor, audio_path, prompt_text=PROMPT_TEXT, max_new_tokens=MAX_NEW_TOKENS):
    audio, sr = sf.read(str(audio_path))
    if audio.ndim > 1:
        audio = audio.mean(axis=1)

    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "audio", "audio_url": str(audio_path)},
                {"type": "text", "text": prompt_text},
            ],
        }
    ]

    text = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
    inputs = processor(text=text, audios=[audio], sampling_rate=sr, return_tensors="pt")
    model_device = next(model.parameters()).device
    inputs = {k: v.to(model_device) if hasattr(v, "to") else v for k, v in inputs.items()}

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            use_cache=True,
        )

    generated_ids = generated_ids[:, inputs["input_ids"].size(1):]
    pred = processor.batch_decode(generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0].strip()
    return pred


def run_inference(model, df_input, run_name, max_new_tokens=MAX_NEW_TOKENS):
    preds = []
    start = time.time()
    for _, row in tqdm(df_input.iterrows(), total=len(df_input), desc=run_name):
        pred = generate_translation(model, processor, Path(row["audio_path"]), max_new_tokens=max_new_tokens)
        preds.append(pred)
    elapsed = time.time() - start
    out_df = df_input.copy()
    out_df["prediction"] = preds
    return out_df, elapsed


def compute_comet(pred_df, comet_model, batch_size=COMET_BATCH_SIZE):
    comet_data = [
        {"src": r["source_en"], "mt": r["prediction"], "ref": r["target_zh"]}
        for _, r in pred_df.iterrows()
    ]
    comet_out = comet_model.predict(comet_data, batch_size=batch_size, gpus=1)
    return comet_out.system_score if hasattr(comet_out, "system_score") else comet_out[1]


# ============================================================
# APPLY FIXED SELECTED-LAYER CODEC
# ============================================================

def make_pipeline_for_run():
    return CodecPipeline(
        filters=[Quantize(digits=CODEC_DIGITS, dtype="f4", astype="f2")],
        compressor=Zstd(level=CODEC_ZSTD_LEVEL),
        encoded_dtype="f2",
    )


def apply_selected_layer_codec(model, subset: str = LAYER_SUBSET):
    selected_layers = get_selected_linear_layer_names(model, subset=subset)
    pipeline = make_pipeline_for_run()
    desc = f"Selected-layer codec: subset={subset}, Quantize(digits={CODEC_DIGITS}, astype='f2') + Zstd"

    applied_rows = []
    for layer_name in tqdm(selected_layers, desc="apply_selected_layer_codec"):
        layer = get_module_by_name(model, layer_name)
        if not isinstance(layer, nn.Linear):
            continue
        compressed_layer = CompressedLinear(layer, pipeline, desc)
        compressed_layer = compressed_layer.to(next(model.parameters()).device)
        set_module_by_name(model, layer_name, compressed_layer)
        applied_rows.append({
            "layer_name": layer_name,
            "layer_subset": subset,
            "compression_method": desc,
        })
    return selected_layers, pd.DataFrame(applied_rows)


# ============================================================
# LOAD MODEL
# ============================================================

baseline_model = Qwen2AudioForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
baseline_model.eval()


# ============================================================
# RECORD ORIGINAL STORAGE
# ============================================================

# Freeze the original linear layer names before any replacement.
original_linear_layer_names = get_linear_layer_names(baseline_model)

original_linear_bytes = {}
for layer_name in original_linear_layer_names:
    layer = get_module_by_name(baseline_model, layer_name)
    if isinstance(layer, nn.Linear):
        original_linear_bytes[layer_name] = estimate_linear_storage_bytes(layer)


# ============================================================
# APPLY COMPRESSION
# ============================================================

selected_layers, applied_df = apply_selected_layer_codec(baseline_model, subset=LAYER_SUBSET)
selected_layers_set = set(selected_layers)

applied_path = EXP_DIR / "selected_layer_codec_policy.csv"
applied_df.to_csv(applied_path, index=False)
selected_layers_path = EXP_DIR / "selected_layers.json"
with open(selected_layers_path, "w", encoding="utf-8") as f:
    json.dump(selected_layers, f, indent=2)

print("selected layers count:", len(selected_layers))
print("policy saved to:", applied_path)


# ============================================================
# STORAGE REPORT
# ============================================================

storage_rows = []
total_original = 0
total_compressed = 0

# IMPORTANT: iterate over the original frozen layer list, not the current nn.Linear list,
# because selected layers have now been replaced by CompressedLinear.
for layer_name in original_linear_layer_names:
    layer = get_module_by_name(baseline_model, layer_name)
    orig_bytes = original_linear_bytes.get(layer_name, np.nan)
    comp_bytes = estimate_linear_storage_bytes(layer)

    if not np.isnan(orig_bytes):
        total_original += orig_bytes
    total_compressed += comp_bytes

    storage_rows.append(
        {
            "layer_name": layer_name,
            "is_selected_layer": layer_name in selected_layers_set,
            "original_bytes": orig_bytes,
            "compressed_bytes": comp_bytes,
            "compression_ratio": (orig_bytes / comp_bytes) if (not np.isnan(orig_bytes) and comp_bytes > 0) else np.nan,
            "method": getattr(layer, "description", "original"),
            "module_type": type(layer).__name__,
        }
    )

storage_report_df = pd.DataFrame(storage_rows)
storage_report_path = EXP_DIR / "selected_layer_codec_storage_report.csv"
storage_report_df.to_csv(storage_report_path, index=False)

storage_totals = {
    "original_total_linear_bytes_est": total_original,
    "compressed_total_linear_bytes_est": total_compressed,
    "compression_ratio_est": total_original / total_compressed if total_compressed > 0 else np.nan,
    "size_scope": "estimated_linear_layers",
    "model_size_on_disk_gb": np.nan,
    "compression_ratio_disk": np.nan,
}

print("storage report saved to:", storage_report_path)
print(storage_totals)


# ============================================================
# EVAL
# ============================================================

experiment_model = baseline_model
experiment_model.eval()

pred_df, eval_time = run_inference(
    experiment_model,
    eval_df,
    run_name=f"selected_layer_codec_{LAYER_SUBSET}_q{CODEC_DIGITS}_eval",
    max_new_tokens=MAX_NEW_TOKENS,
)

preds_path = EXP_DIR / "selected_layer_codec_eval_preds.csv"
pred_df.to_csv(preds_path, index=False, encoding="utf-8")
comet_score = compute_comet(pred_df, comet_model)

print("predictions saved to:", preds_path)
print("eval seconds:", round(eval_time, 2))
print("eval COMET:", comet_score)


# ============================================================
# SUMMARY
# ============================================================

timestamp_utc = pd.Timestamp.utcnow().isoformat()

summary_df = pd.DataFrame([
    {
        "timestamp_utc": timestamp_utc,
        "experiment_name": EXPERIMENT_NAME,
        "run": f"selected_layer_codec_{LAYER_SUBSET}_q{CODEC_DIGITS}_eval",
        "split": "eval",
        "rows": len(pred_df),
        "seconds": eval_time,
        "seconds_per_item": eval_time / len(pred_df),
        "comet": comet_score,
        "model_id": MODEL_ID,
        "prompt_text": PROMPT_TEXT,
        "max_new_tokens": MAX_NEW_TOKENS,
        "comet_batch_size": COMET_BATCH_SIZE,
        "predictions_csv": str(preds_path),
        "quantization_method": "custom_selected_linear_codec",
        "precision": "custom",
        "layer_subset": LAYER_SUBSET,
        "codec_digits": CODEC_DIGITS,
        "codec_zstd_level": CODEC_ZSTD_LEVEL,
        "selected_layer_count": len(selected_layers),
        "original_total_linear_bytes_est": storage_totals["original_total_linear_bytes_est"],
        "compressed_total_linear_bytes_est": storage_totals["compressed_total_linear_bytes_est"],
        "compression_ratio_est": storage_totals["compression_ratio_est"],
        "size_scope": storage_totals["size_scope"],
        "model_size_on_disk_gb": storage_totals["model_size_on_disk_gb"],
        "compression_ratio_disk": storage_totals["compression_ratio_disk"],
        "compression_method": f"Selected-layer codec subset={LAYER_SUBSET} q{CODEC_DIGITS} + Zstd",
        "selected_layers_json": str(selected_layers_path),
        "storage_report_csv": str(storage_report_path),
    }
])

summary_path = EXP_DIR / "selected_layer_codec_eval_summary.csv"
summary_df.to_csv(summary_path, index=False)

print("summary saved to:", summary_path)
print(summary_df)


# ============================================================
# CLEANUP
# ============================================================

del experiment_model
cleanup_cuda()


project root exists: True
cuda available: True
gpu: NVIDIA RTX A6000
experiment: custom_selected_linear_codec_all_selected_q3_zstd_eval_only_en_zh
codec digits: 3
layer subset: all_selected


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

/home/paperspace/miniconda3/envs/iwslt-2026/lib/python3.11/site-packages/lightning_fabric/utilities/cloud_io.py:52: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torc

eval rows: 416


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

apply_selected_layer_codec:   0%|          | 0/96 [00:00<?, ?it/s]

selected layers count: 96
policy saved to: /home/paperspace/projects/iwslt2026-compression/outputs/experiment_results/custom_selected_linear_codec_all_selected_q3_zstd_eval_only_en_zh/selected_layer_codec_policy.csv
storage report saved to: /home/paperspace/projects/iwslt2026-compression/outputs/experiment_results/custom_selected_linear_codec_all_selected_q3_zstd_eval_only_en_zh/selected_layer_codec_storage_report.csv
{'original_total_linear_bytes_est': 15500451840, 'compressed_total_linear_bytes_est': 11072301600, 'compression_ratio_est': 1.3999304209704693, 'size_scope': 'estimated_linear_layers', 'model_size_on_disk_gb': nan, 'compression_ratio_disk': nan}


selected_layer_codec_all_selected_q3_eval:   0%|          | 0/416 [00:00<?, ?it/s]

/home/paperspace/miniconda3/envs/iwslt-2026/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/paperspace/miniconda3/envs/iwslt-2026/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.5` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/home/paperspace/miniconda3/envs/iwslt-2026/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:650: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
We

predictions saved to: /home/paperspace/projects/iwslt2026-compression/outputs/experiment_results/custom_selected_linear_codec_all_selected_q3_zstd_eval_only_en_zh/selected_layer_codec_eval_preds.csv
eval seconds: 312.7
eval COMET: 0.7766893485274452
summary saved to: /home/paperspace/projects/iwslt2026-compression/outputs/experiment_results/custom_selected_linear_codec_all_selected_q3_zstd_eval_only_en_zh/selected_layer_codec_eval_summary.csv
                      timestamp_utc  \
0  2026-03-20T15:35:24.055689+00:00   

                                     experiment_name  \
0  custom_selected_linear_codec_all_selected_q3_z...   

                                         run split  rows     seconds  \
0  selected_layer_codec_all_selected_q3_eval  eval   416  312.698004   

   seconds_per_item     comet                      model_id  \
0          0.751678  0.776689  Qwen/Qwen2-Audio-7B-Instruct   

                                   prompt_text  ...  selected_layer_count  \
0  Translate